In [0]:
%run ./utility/logger

In [0]:
dbutils.widgets.text('catalog',"")
dbutils.widgets.text('schema',"")
dbutils.widgets.text('env',"")

In [0]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
env = dbutils.widgets.get('env')

In [0]:
print(schema)

In [0]:
from pyspark.sql.functions import *

start_date = "2020-01-01"
end_date = "2035-12-31"

date_df = spark.sql(f"""
SELECT explode(
    sequence(
        to_date('{start_date}'),
        to_date('{end_date}'),
        interval 1 day
    )
) AS full_date
""")

date_df = (
    date_df
    .withColumn("date_key", date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", year("full_date").cast("int"))
    .withColumn("quarter", quarter("full_date").cast("int"))
    .withColumn("month", month("full_date").cast("int"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("week_of_year", weekofyear("full_date").cast("int"))
    .withColumn("day_of_month", dayofmonth("full_date").cast("int"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
)

date_df = date_df.select(
    "date_key",
    "full_date",
    "year",
    "quarter",
    "month",
    "month_name",
    "week_of_year",
    "day_of_month",
    "day_name"
)

date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.dim_date")